# WIDS: Deauth, Disassociation and Reassociation Detection (AWID3)

Prototype Wi-Fi intrusion detector using a subset of AWID3. This notebook analyses three management-frame attacks: deauthentication, disassociation and (re)association.

In [2]:
import os, re
import pandas as pd
import numpy as np

root = '/kaggle/input/datasets/suumia/awid3-dataset/CSV'
for d in sorted(os.listdir(root)):
    n = len(os.listdir(os.path.join(root, d)))
    print(d, "->", n, "files")

1.Deauth -> 33 files
10.SQL_Injection -> 52 files
11.SSDP -> 163 files
12.Evil_Twin -> 76 files
13.Website_spoofing -> 54 files
2.Disas -> 41 files
3.(Re)Assoc -> 37 files
4.Rogue_AP -> 40 files
5.Krack -> 29 files
6.Kr00k -> 59 files
7.SSH -> 49 files
8.Botnet -> 65 files
9.Malware -> 47 files


## 1. Dataset structure

AWID3 is organised as one folder per attack (13 in total). Each folder holds numbered CSV parts of roughly 50,000 frames, in capture order. This notebook uses three of them: `1.Deauth`, `2.Disas` and `3.(Re)Assoc`. The others (Evil_Twin, Rogue_AP, Krack and so on) are left for possible future work.

In [3]:
attack_dirs = ["1.Deauth", "2.Disas", "3.(Re)Assoc"]   

def part_no(f):
    return int(re.findall(r"\d+", f)[-1])

label_counts = {}   
for d in attack_dirs:
    path = os.path.join(root, d)
    label_counts[d] = {}
    for f in sorted(os.listdir(path), key=part_no):
        lab = pd.read_csv(os.path.join(path, f), usecols=["Label"])["Label"]
        label_counts[d][f] = lab.value_counts().to_dict()

for d, parts in label_counts.items():
    total = sum(sum(c.values()) for c in parts.values())
    attack = sum(v for c in parts.values() for k, v in c.items() if k != "Normal")
    print(d, "| parts:", len(parts), "| frames:", total, "| attack frames:", attack)

1.Deauth | parts: 33 | frames: 1626469 | attack frames: 38942
2.Disas | parts: 41 | frames: 2013716 | attack frames: 75131
3.(Re)Assoc | parts: 37 | frames: 1843932 | attack frames: 5502


## 2. Label scan

The files are too large to load fully, so only the `Label` column is read from every part. This gives the number of frames and attack frames per attack folder without using much memory.

### Findings: size and class balance

| Attack | Parts | Frames | Attack frames | Attack share |
|---|---|---|---|---|
| Deauth | 33 | 1,626,469 | 38,942 | ~2.4% |
| Disas | 41 | 2,013,716 | 75,131 | ~3.7% |
| (Re)Assoc | 37 | 1,843,932 | 5,502 | ~0.3% |

- The data is heavily imbalanced: attacks are a small share of frames, and (Re)Assoc is the rarest.
- Plain accuracy would therefore be misleading (predicting "Normal" everywhere already scores above 96%). Precision, recall and macro-F1 will be reported alongside it.

In [4]:
for d, parts in label_counts.items():
    useful = [f for f, c in parts.items() if any(k != "Normal" for k in c)]
    print(d, "-> parts with attacks:", useful)
    print("   label names:", {k for c in parts.values() for k in c})

1.Deauth -> parts with attacks: ['Deauth_21.csv', 'Deauth_22.csv', 'Deauth_23.csv', 'Deauth_24.csv', 'Deauth_25.csv', 'Deauth_26.csv', 'Deauth_27.csv', 'Deauth_28.csv', 'Deauth_29.csv', 'Deauth_30.csv', 'Deauth_31.csv', 'Deauth_32.csv']
   label names: {'Normal', 'Deauth'}
2.Disas -> parts with attacks: ['Disas_28.csv', 'Disas_29.csv', 'Disas_30.csv', 'Disas_31.csv', 'Disas_32.csv', 'Disas_33.csv', 'Disas_34.csv', 'Disas_35.csv', 'Disas_36.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_39.csv', 'Disas_40.csv']
   label names: {'Normal', 'Disas'}
3.(Re)Assoc -> parts with attacks: ['(Re)Assoc_22.csv', '(Re)Assoc_23.csv', '(Re)Assoc_24.csv', '(Re)Assoc_25.csv', '(Re)Assoc_26.csv', '(Re)Assoc_27.csv', '(Re)Assoc_28.csv', '(Re)Assoc_29.csv', '(Re)Assoc_30.csv', '(Re)Assoc_31.csv', '(Re)Assoc_32.csv', '(Re)Assoc_33.csv', '(Re)Assoc_34.csv', '(Re)Assoc_35.csv', '(Re)Assoc_36.csv']
   label names: {'Normal', '(Re)Assoc'}


### Findings: where the attacks are

- Each file contains only `Normal` plus its own attack label (`Deauth`, `Disas` or `(Re)Assoc`), so every folder is a two-class problem.
- Attack frames appear only in the later parts: Deauth 21–32, Disas 28–40, (Re)Assoc 22–36. The earlier parts are normal-only.
- The part numbers follow capture time, so a time-based split is natural. A random frame-level split would leak, because neighbouring frames are almost identical.
- Only the parts containing attacks, plus a few normal-only parts, will be loaded.

In [5]:
p = pd.read_csv(os.path.join(root, "1.Deauth", "Deauth_22.csv"), low_memory=False)

for key in ["wlan.fc", "wlan.sa", "wlan.da", "wlan.bssid", "wlan.ta", "wlan.ra",
            "antsignal", "wlan.fixed.reason"]:
    print(key, [c for c in p.columns if key in c])

print(pd.crosstab(p["Label"], [p["wlan.fc.type"], p["wlan.fc.subtype"]]))

wlan.fc ['wlan.fc.ds', 'wlan.fc.frag', 'wlan.fc.order', 'wlan.fc.moredata', 'wlan.fc.protected', 'wlan.fc.pwrmgt', 'wlan.fc.type', 'wlan.fc.retry', 'wlan.fc.subtype', 'wlan.fcs.bad_checksum']
wlan.sa ['wlan.sa']
wlan.da ['wlan.da']
wlan.bssid ['wlan.bssid']
wlan.ta ['wlan.ta', 'wlan.tag', 'wlan.tag.length']
wlan.ra ['wlan.ra']
antsignal ['radiotap.dbm_antsignal']
wlan.fixed.reason ['wlan.fixed.reason_code']
wlan.fc.type     0                                         ...    1        \
wlan.fc.subtype 0  1   2   3    4   5    8   10  11    12  ...   8     9    
Label                                                      ...              
Deauth           0  0   0   0    0   0    0  14   0  5187  ...    0     0   
Normal           4  3  18  19  128  53  214  22  44     1  ...  462  5227   

wlan.fc.type                          2                    
wlan.fc.subtype     11    12    13   0   4      8  9   12  
Label                                                      
Deauth               0 

In [6]:
atk = p[p["Label"] == "Deauth"]
for c in ["wlan.sa", "wlan.da", "wlan.bssid", "wlan.fixed.reason_code"]:
    print(c)
    print(atk[c].value_counts().head(5))
    print()
print(atk["radiotap.dbm_antsignal"].describe())

wlan.sa
wlan.sa
0c:9d:92:54:fe:34    2643
b4:ce:40:c8:33:81    1332
88:66:a5:55:a2:d4    1030
94:e9:79:82:c5:77     196
Name: count, dtype: int64

wlan.da
wlan.da
0c:9d:92:54:fe:34    2558
b4:ce:40:c8:33:81    1497
88:66:a5:55:a2:d4     922
94:e9:79:82:c5:77     210
00:c0:ca:a8:26:3e      14
Name: count, dtype: int64

wlan.bssid
wlan.bssid
0c:9d:92:54:fe:34    5201
Name: count, dtype: int64

wlan.fixed.reason_code
wlan.fixed.reason_code
7.0    4332
1.0     671
6.0     196
3.0       2
Name: count, dtype: int64

count    5201.000000
mean     -133.683522
std        18.527937
min      -225.000000
25%      -130.000000
50%      -129.000000
75%      -126.000000
max       -99.000000
Name: radiotap.dbm_antsignal, dtype: float64


In [7]:
print(p.info())
print(p.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 254 entries, frame.encap_type to Label
dtypes: float64(115), int64(24), object(115)
memory usage: 96.9+ MB
None
   frame.encap_type  frame.len  frame.number  \
0                23         88       1100001   
1                23        260       1100002   
2                23         88       1100003   
3                23         76       1100004   
4                23         70       1100005   

                                          frame.time  frame.time_delta  \
0  Dec 18- 2020 17:50:06.288731000 GTB Standard Time          0.000042   
1  Dec 18- 2020 17:50:06.289026000 GTB Standard Time          0.000295   
2  Dec 18- 2020 17:50:06.289034000 GTB Standard Time          0.000008   
3  Dec 18- 2020 17:50:06.289136000 GTB Standard Time          0.000102   
4  Dec 18- 2020 17:50:06.289140000 GTB Standard Time          0.000004   

   frame.time_delta_displayed  frame.time_epoch  frame.time_relative 

In [ ]:
mask = (p["wlan.fc.type"] == 0) & (p["wlan.fc.subtype"] == 12)
deauth = p[mask]

print(len(deauth))
deauth[["frame.time_epoch", "wlan.sa", "wlan.da", "wlan.fixed.reason_code", "Label"]].head(10)

In [19]:
print(deauth.groupby('frame.time_epoch').size())

frame.time_epoch
1608306606     13
1608306607    263
1608306608    236
1608306609    289
1608306610    297
1608306611    200
1608306612    245
1608306613    202
1608306614    169
1608306615    254
1608306616    230
1608306617    228
1608306618    253
1608306619    239
1608306620    231
1608306621    265
1608306622    213
1608306623    222
1608306624    254
1608306625    226
1608306626    211
1608306627    274
1608306628    174
dtype: int64
